# Raw video VideoMAE experiment

기존 clip 실험은 그대로 두고, 원본 10초 영상을 자르지 않은 상태로 VideoMAE 학습을 확인한다.

실행 순서:
1. 라벨별 50개 원본 영상 학습
2. 결과가 정상일 때 라벨별 100개 원본 영상 학습

서비스 입력 정책:
- 사용자 입력 영상은 5초 이상 30초 이하만 받는다.
- 30초 초과 영상은 사고 지점이 포함된 5~30초 구간으로 다시 제출하도록 요청한다.


In [1]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter, defaultdict

RUN_INSTALL_REQUIREMENTS = True
RUN_DOWNLOAD = True
RUN_TRAIN_50 = True
RUN_TRAIN_100 = False  # 50개 실험 확인 후 True로 변경

PROJECT_ROOT = Path(r'D:\dev\SKN27-FINAL-3Team')
if not PROJECT_ROOT.exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'ai').exists() and (candidate / 'etl').exists() and (candidate / 'storage').exists():
            PROJECT_ROOT = candidate
            break

MANIFEST_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/manifests'
RAW_VIDEO_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/raw_videos'
MODEL_DIR = PROJECT_ROOT / 'storage/vision/models/videomae_raw_video'
SAMPLE_MANIFEST = MANIFEST_DIR / 'sample_700_coarse_manifest.csv'
FULL_DOWNLOAD_MANIFEST = MANIFEST_DIR / 'train_700_download_manifest.csv'

FRAME_COUNT = 16
BATCH_SIZE = 1
LEARNING_RATE = 0.0001
EPOCHS = 5
SEED = 42
DEVICE = 'auto'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SAMPLE_MANIFEST:', SAMPLE_MANIFEST)
print('FULL_DOWNLOAD_MANIFEST:', FULL_DOWNLOAD_MANIFEST)


PROJECT_ROOT: /workspace/SKN27-FINAL-3Team
SAMPLE_MANIFEST: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/sample_700_coarse_manifest.csv
FULL_DOWNLOAD_MANIFEST: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_download_manifest.csv


In [2]:
def run_command(command, *, enabled=True, timeout=None):
    command = list(map(str, command))
    print('$', ' '.join(command), flush=True)
    if not enabled:
        print('SKIPPED')
        return None
    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        encoding='utf-8',
        errors='replace',
        timeout=timeout,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed


def read_csv(path):
    with Path(path).open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def write_csv(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row.keys()))
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def resolve_local_video_path(path_value):
    if not path_value:
        return None
    path_text = str(path_value).replace('\\', '/')
    runpod_prefix = '/workspace/SKN27-FINAL-3Team/'
    if path_text.startswith(runpod_prefix):
        return PROJECT_ROOT / path_text[len(runpod_prefix):]
    path = Path(path_value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def make_subset_manifest(source_manifest, output_manifest, per_label):
    rows = read_csv(source_manifest)
    selected = []
    counts = defaultdict(int)
    for row in rows:
        label = row.get('coarse_label') or row.get('label')
        if not label or counts[label] >= per_label:
            continue
        path = resolve_local_video_path(row.get('local_path') or row.get('file_path'))
        if path is not None:
            if not path.exists():
                continue
            row = dict(row)
            row['local_path'] = str(path)
            row['file_exists'] = 'True'
        selected.append(row)
        counts[label] += 1
    write_csv(selected, output_manifest)
    print('subset_manifest:', output_manifest)
    print('rows:', len(selected))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in selected)))
    return output_manifest


def latest_run_dir(path):
    runs = [p for p in Path(path).glob('videomae_cls_*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No runs found under {path}')
    return sorted(runs)[-1]


In [3]:
if not SAMPLE_MANIFEST.exists() and not FULL_DOWNLOAD_MANIFEST.exists():
    raise FileNotFoundError(f'Missing source manifest: {SAMPLE_MANIFEST}')

if not FULL_DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        'etl/vision/download_sampled_media.py',
        '--input', SAMPLE_MANIFEST,
        '--output', FULL_DOWNLOAD_MANIFEST,
        '--download-dir', RAW_VIDEO_DIR,
        '--label-column', 'coarse_label',
        '--per-label', '700',
        '--split', '',
    ], enabled=RUN_DOWNLOAD, timeout=None)

rows = read_csv(FULL_DOWNLOAD_MANIFEST)
print('download_rows:', len(rows))
print('label_counts:', dict(Counter(row.get('coarse_label') for row in rows)))
print('download_status:', dict(Counter(row.get('download_status') for row in rows)))
print('file_exists:', dict(Counter(row.get('file_exists') for row in rows)))


FileNotFoundError: Missing source manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/sample_700_coarse_manifest.csv

In [ ]:
MANIFEST_50 = MANIFEST_DIR / 'train_50_raw_video_manifest.csv'
make_subset_manifest(FULL_DOWNLOAD_MANIFEST, MANIFEST_50, per_label=50)


In [ ]:
EXPERIMENT_50 = {
    'name': 'raw_video_50_per_label_fc16',
    'manifest': MANIFEST_50,
    'output_dir': MODEL_DIR / 'per_label_50',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': True,
}

command = [
    sys.executable,
    'ai/vision/train_videomae_classifier.py',
    '--manifest', EXPERIMENT_50['manifest'],
    '--root-dir', PROJECT_ROOT,
    '--output-dir', EXPERIMENT_50['output_dir'],
    '--label-column', 'coarse_label',
    '--frame-count', EXPERIMENT_50['frame_count'],
    '--epochs', EXPERIMENT_50['epochs'],
    '--batch-size', EXPERIMENT_50['batch_size'],
    '--learning-rate', EXPERIMENT_50['learning_rate'],
    '--seed', SEED,
    '--device', DEVICE,
    '--num-workers', '0',
    '--no-show-progress',
    '--freeze-backbone',
]
run_command(command, enabled=RUN_TRAIN_50, timeout=None)
if RUN_TRAIN_50:
    print('LAST_RUN_DIR_50:', latest_run_dir(EXPERIMENT_50['output_dir']))


In [ ]:
MANIFEST_100 = MANIFEST_DIR / 'train_100_raw_video_manifest.csv'
make_subset_manifest(FULL_DOWNLOAD_MANIFEST, MANIFEST_100, per_label=100)


In [ ]:
EXPERIMENT_100 = {
    'name': 'raw_video_100_per_label_fc16',
    'manifest': MANIFEST_100,
    'output_dir': MODEL_DIR / 'per_label_100',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': True,
}

command = [
    sys.executable,
    'ai/vision/train_videomae_classifier.py',
    '--manifest', EXPERIMENT_100['manifest'],
    '--root-dir', PROJECT_ROOT,
    '--output-dir', EXPERIMENT_100['output_dir'],
    '--label-column', 'coarse_label',
    '--frame-count', EXPERIMENT_100['frame_count'],
    '--epochs', EXPERIMENT_100['epochs'],
    '--batch-size', EXPERIMENT_100['batch_size'],
    '--learning-rate', EXPERIMENT_100['learning_rate'],
    '--seed', SEED,
    '--device', DEVICE,
    '--num-workers', '0',
    '--no-show-progress',
    '--freeze-backbone',
]
run_command(command, enabled=RUN_TRAIN_100, timeout=None)
if RUN_TRAIN_100:
    print('LAST_RUN_DIR_100:', latest_run_dir(EXPERIMENT_100['output_dir']))


In [ ]:
def print_latest_history(output_dir):
    run_dir = latest_run_dir(output_dir)
    print('run_dir:', run_dir)
    for file_name in ['run_config.json', 'training_history.csv']:
        path = run_dir / file_name
        print('##', file_name, path.exists())
        if path.suffix == '.json' and path.exists():
            print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:3000])
        elif path.exists():
            for row in read_csv(path):
                print(row)

if RUN_TRAIN_50:
    print_latest_history(EXPERIMENT_50['output_dir'])
if RUN_TRAIN_100:
    print_latest_history(EXPERIMENT_100['output_dir'])
